# Workshop RAG Completo

Neste notebook, vamos construir um pipeline de RAG (Retrieval-Augmented Generation) do zero. Os passos envolvem:
1. Setup e instalação de dependências
2. Ingestão de um documento (usando `markitdown`)
3. Chunking do texto convertido (usando `MarkdownTextSplitter`)
4. Criação de embeddings e armazenamento no **Qdrant Cloud**
5. Busca Vetorial (Similarity Search)
6. Resposta gerada via LLM conectada aos documentos

## 1. Setup Inicial
Primeiro, vamos instalar as dependências necessárias.

In [1]:
%pip install -q markitdown[all] langchain langchain-openai langchain-nvidia-ai-endpoints langchain-qdrant qdrant-client python-dotenv langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


Agora, vamos carregar nossas variáveis de ambiente. Certifique-se de ter criado um arquivo `.env` na raiz do projeto (baseado no `.env.example`) e preenchido suas credenciais.

In [2]:
import os
from dotenv import load_dotenv

# Carrega as variáveis de ambiente do arquivo .env
load_dotenv()

# Verificando se foram carregadas corretamente (sem expor as chaves completas)
print("OPENAI_API_KEY carregada:", bool(os.getenv("OPENAI_API_KEY")))
print("OPENAI_API_BASE configurada:", os.getenv("OPENAI_API_BASE"))
print("QDRANT_URL configurada:", os.getenv("QDRANT_URL"))

OPENAI_API_KEY carregada: True
OPENAI_API_BASE configurada: https://integrate.api.nvidia.com/v1/
QDRANT_URL configurada: https://a6b27eba-fb43-478b-8311-419c6de95f66.sa-east-1-0.aws.cloud.qdrant.io


## 2. Ingestão e Conversão de Documentos
Utilizaremos o `markitdown` da Microsoft para extrair o texto de arquivos (como PDFs, Word, Markdown, etc.) de forma robusta e gerar um Markdown limpo.
**Nota:** Certifique-se de que o arquivo `docs/documento_exemplo.md` exista.

In [3]:
from markitdown import MarkItDown

md = MarkItDown()

document_path = "docs/D60282026.pdf" # O caminho relativo ao notebook
result = md.convert(document_path)

markdown_content = result.text_content
print(f"Tamanho do texto extraído: {len(markdown_content)} caracteres.\n")
print("Prévia do conteúdo:\n")
print(markdown_content[:500] + "...")

Tamanho do texto extraído: 4950 caracteres.

Prévia do conteúdo:

Prefeitura Municipal de Lagoa Santa

DECRETO Nº 6.028, DE 07 DE MAIO DE 2026.

Aprova  o  loteamento  denominado  "Bellare”,
situado  na  Gleba  01  do  local  denominado
“Fazenda  Zumbi”,  neste  Município  de  Lagoa
Santa/MG,
Emccamp
Incorporação  SC  03  SPE  Ltda.,  e  dá  outras
providências.

propriedade

de

O  PREFEITO  DO  MUNICÍPIO  DE  LAGOA  SANTA,  no  uso  e  gozo  das
atribuições que lhe confere o art. 68 da Lei Orgânica Municipal, de acordo com as disposições
da  Lei  Federal  nº...


## 3. Chunking e Embeddings
Como o `markitdown` gera texto em Markdown, usar o `MarkdownTextSplitter` do LangChain é ideal, pois ele tenta preservar a estrutura dos cabeçalhos durante a divisão do texto.

In [4]:
from langchain_text_splitters import MarkdownTextSplitter
from langchain_core.documents import Document

# Configurando o separador de texto
splitter = MarkdownTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

# Como temos apenas uma string enorme, transformamos em uma lista de strings (chunks)
chunks_text = splitter.split_text(markdown_content)

# Convertendo em objetos Document do Langchain para facilitar a injeção no vector db
docs = [Document(page_content=chunk, metadata={"source": document_path}) for chunk in chunks_text]

print(f"Total de chunks gerados: {len(docs)}")
print("Exemplo do primeiro chunk:\n", docs[0].page_content)

Total de chunks gerados: 6
Exemplo do primeiro chunk:
 Prefeitura Municipal de Lagoa Santa

DECRETO Nº 6.028, DE 07 DE MAIO DE 2026.

Aprova  o  loteamento  denominado  "Bellare”,
situado  na  Gleba  01  do  local  denominado
“Fazenda  Zumbi”,  neste  Município  de  Lagoa
Santa/MG,
Emccamp
Incorporação  SC  03  SPE  Ltda.,  e  dá  outras
providências.

propriedade

de

O  PREFEITO  DO  MUNICÍPIO  DE  LAGOA  SANTA,  no  uso  e  gozo  das
atribuições que lhe confere o art. 68 da Lei Orgânica Municipal, de acordo com as disposições
da  Lei  Federal  nº  6.766,  de  19  de  dezembro  de  1979,  Lei  Municipal  nº  2.759,  de  28  de
dezembro de 2007; e

Considerando  os  pareceres  técnicos,  Certidão  de  Conformidade  Urbanística  e
esclarecimentos  técnicos  emitidos  pela  atual  Secretaria  Municipal  de  Infraestrutura  e  Meio
Ambiente  nos  Processos  Administrativos  Eletrônicos  nº  4271/2017  e  nº  3400/2025,  todos
favoráveis à implantação do loteamento na área;


## 4. Armazenamento no Qdrant Cloud
Agora criamos nossos embeddings (representações vetoriais) e os subimos para o nosso banco de dados vetorial Qdrant Cloud.

In [11]:
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_qdrant import QdrantVectorStore

# Inicializamos o modelo de Embeddings. 
# Ele utilizará as variáveis OPENAI_API_KEY e OPENAI_API_BASE configuradas anteriormente.
embeddings = NVIDIAEmbeddings(model="nvidia/llama-3.2-nemoretriever-300m-embed-v1")

url = os.getenv("QDRANT_URL")
api_key = os.getenv("QDRANT_API_KEY")
collection_name = "workshop_rag"

# Envia os documentos, calculando os embeddings no processo
vector_store = QdrantVectorStore.from_documents(
    docs,
    embeddings,
    url=url,
    api_key=api_key,
    collection_name=collection_name,
    force_recreate=True # Apenas para o workshop: apaga os dados antigos e recria a coleção
)

print("Documentos indexados com sucesso no Qdrant Cloud!")

/home/lincolnminto/Projects/ai-heroes/.venv/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found nvidia/llama-3.2-nemoretriever-300m-embed-v1 in available_models, but type is unknown and inference may fail.
  warnings.warn(


Documentos indexados com sucesso no Qdrant Cloud!


## 5. Busca Vetorial (Similarity Search)
Antes de pedirmos para a IA responder, vamos ver como a busca funciona nos bastidores. 
A ideia aqui é achar no banco de dados quais `chunks` são mais parecidos com a nossa pergunta.

In [21]:
query = "Qual é o tema principal deste documento?"

# Retorna os 3 documentos mais relevantes
resultados_busca = vector_store.similarity_search(query, k=3)

print(f"Encontrados {len(resultados_busca)} resultados para a query: '{query}'\n")
for i, doc in enumerate(resultados_busca):
    print(f"--- Resultado {i+1} ---")
    print(doc.page_content[:300] + "...\n")

Encontrados 3 resultados para a query: 'Qual é o tema principal deste documento?'

--- Resultado 1 ---
2. área verde 2 medindo 13.097,92m²;

c) sistema viário medindo 8.673,64m².

Art. 2º O zoneamento do loteamento “Bellare” fica definido como:

Quadras

1

2

3

3

4

Ruas

Coletora 1

Coletora 1

Coletora 1

Coletora 1

Coletora 1

Lotes

1 a 5

1 a 5

1

2

1 a 3

Zoneamentos
C-3

C-3

C-3

SE-3

...

--- Resultado 2 ---
Art.  4º  Todos  os  ônus  decorrentes  da  execução  das  obras  de  implantação  do
loteamento,  das  medidas  mitigadoras  e  compensatórias,  assim  como  quaisquer  gastos  ou
despesas  oriundos  desta  aprovação,  incluídos  emolumentos,  tributos,  registros  e  custos
relativos  ao  Termo  d...

--- Resultado 3 ---
Considerando o Selo de Exame e Anuência Prévia da Agência de Desenvolvimento da
Região  Metropolitana  de  Belo  Horizonte  (ARMBH)  aposta  na  planta  urbanística  do
loteamento e a Certidão de Anuência Prévia Metropolitana nº 39/2024, vinculada

## 6. Geração Aumentada (RAG Chain)
Finalmente, vamos juntar a busca do Qdrant com um LLM, criando uma *Retrieval Chain* no LangChain.

In [27]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from pydantic import SecretStr

def format_docs(docs):
    return "\\n\\n".join(doc.page_content for doc in docs)

# 1. Instanciamos o LLM. Ele usará as credenciais de OPENAI_API_KEY e OPENAI_API_BASE
llm = ChatOpenAI(base_url=os.getenv("OPENAI_API_BASE"), api_key=SecretStr(os.getenv("OPENAI_API_KEY", "")), model="openai/gpt-oss-20b", temperature=0)

# 2. Criamos o Prompt que ditará como o modelo deve agir com o contexto recuperado
system_prompt = (
    """
        Você é um assistente prestativo. Use os trechos de contexto recuperados a seguir 
        para responder à pergunta. Se não souber a resposta, diga que não sabe. Use apenas o contexto abaixo fornecido para responder as perguntas do usuário.
        Contexto: {context}
    """
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# 3. Criamos o retriever (a ferramenta que busca os documentos no vector store)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# 4. Montamos a RAG Chain (compatível com LangChain 1.x)
rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG Chain configurada com sucesso!")

RAG Chain configurada com sucesso!


Agora é só invocar a chain!

In [28]:
resposta = rag_chain.invoke(query)

print(f"Pergunta: {query}\n")
print("Resposta do LLM:")
print(resposta)

Pergunta: Qual é o tema principal deste documento?

Resposta do LLM:
O documento trata do **zoneamento e das obrigações de implantação do loteamento “Bellare”**, estabelecendo as áreas verdes, vias, lotes, zoneamentos, responsabilidades da incorporadora (Emccamp Incorporação SC 03 SPE Ltda.) e os requisitos legais e administrativos para a execução das obras de infraestrutura, mitigação e compensação.
